# Section 1 — Project Setup

In [ ]:
import os
import sys
import json
import time
import math
from pathlib import Path
from typing import Dict, List, Any, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

NOTEBOOK_DIR = Path.cwd() if Path.cwd().name == 'notebooks' else Path.cwd() / 'Phase_2_Deep_Learning' / 'notebooks'
PHASE2_DIR = NOTEBOOK_DIR.parent
PROJECT_ROOT = PHASE2_DIR.parent

if str(PHASE2_DIR) not in sys.path:
    sys.path.insert(0, str(PHASE2_DIR))
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.dl_models import MLPConfig, WaterQualityMLP, build_model, load_model_checkpoint
from src.dataset import get_dataloaders, load_feature_config, get_default_paths
from src.training import (
    get_device,
    get_loss_function,
    get_optimizer,
    get_scheduler,
    step_scheduler,
    compute_regression_metrics,
    EarlyStopping,
    save_training_checkpoint,
    load_training_checkpoint,
    train_one_epoch,
    validate,
    analyze_overfitting,
    format_history_table,
    MLPTrainer,
)

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = get_device('auto')

print('=' * 75)
print('TASK 3: MLP TRAINING & OPTIMIZATION — SETUP COMPLETE')
print('=' * 75)
print(f'PyTorch Version    : {torch.__version__}')
print(f'Selected Device    : {DEVICE} ({DEVICE.type.upper()})')
print(f'Deterministic Seed : {SEED}')
print(f'Project Root       : {PROJECT_ROOT}')
print(f'Phase 2 Root       : {PHASE2_DIR}')

# Section 2 — Data Loading & Dimension Verification

In [ ]:
feature_cfg = load_feature_config()
INPUT_DIM = feature_cfg['total_input_dimension']
BATCH_SIZE = feature_cfg.get('batch_size', 1024)
TARGET_VAR = feature_cfg['target_variable']

print(f'Loading splits and constructing DataLoaders (batch_size={BATCH_SIZE})...')
train_loader, val_loader, test_loader, _ = get_dataloaders(
    batch_size=BATCH_SIZE,
    num_workers=0,
)

print(f'\nDataset Verification:')
print(f'  Total Input Dimension : {INPUT_DIM} (36 numerical + 17 one-hot encoded)')
print(f'  Target Variable       : {TARGET_VAR}')
print(f'  Train Batches         : {len(train_loader):>6,} ({len(train_loader.dataset):>10,} rows)')
print(f'  Val Batches           : {len(val_loader):>6,} ({len(val_loader.dataset):>10,} rows)')
print(f'  Test Batches          : {len(test_loader):>6,} ({len(test_loader.dataset):>10,} rows)')

sample_x, sample_y = next(iter(train_loader))
assert sample_x.shape == (BATCH_SIZE, INPUT_DIM), f'Shape mismatch: {sample_x.shape}'
assert sample_y.shape == (BATCH_SIZE, 1), f'Target shape mismatch: {sample_y.shape}'
assert not torch.isnan(sample_x).any(), 'NaN in features'
assert not torch.isnan(sample_y).any(), 'NaN in targets'
print('\n✓ All DataLoader shape, dimension, and non-null assertions passed.')

# Section 3 — Baseline MLP Training

In [ ]:
mlp_base_config = MLPConfig(
    input_dim=INPUT_DIM,
    hidden_dims=[512, 256, 128, 64],
    dropout_rate=0.30,
    use_batch_norm=True,
    activation='relu',
    output_dim=1,
    name='MLP_Base',
)

baseline_model = build_model(mlp_base_config, device=DEVICE, seed=SEED)
print(baseline_model.architecture_summary())

baseline_criterion = get_loss_function('mse')
baseline_optimizer = get_optimizer('adam', baseline_model, lr=1e-3, weight_decay=1e-4)

baseline_trainer = MLPTrainer(
    model=baseline_model,
    optimizer=baseline_optimizer,
    criterion=baseline_criterion,
    device=DEVICE,
    clip_grad_norm=1.0,
)

EPOCHS_BASELINE = 3
print(f'\nStarting baseline training for {EPOCHS_BASELINE} epochs...')
baseline_history = baseline_trainer.fit(
    train_loader=train_loader,
    val_loader=val_loader,
    epochs=EPOCHS_BASELINE,
    verbose=True,
)

# Section 4 — Loss Function Experiments

In [ ]:
loss_experiment_results = []
LOSS_CANDIDATES = ['mse', 'l1', 'smoothl1']
EPOCHS_LOSS_EXP = 2

for loss_name in LOSS_CANDIDATES:
    print(f'\n--- Training MLP_Base with Loss: {loss_name.upper()} ---')
    model_exp = build_model(mlp_base_config, device=DEVICE, seed=SEED)
    criterion_exp = get_loss_function(loss_name, beta=1.0)
    optimizer_exp = get_optimizer('adam', model_exp, lr=1e-3, weight_decay=1e-4)

    trainer_exp = MLPTrainer(
        model=model_exp,
        optimizer=optimizer_exp,
        criterion=criterion_exp,
        device=DEVICE,
        clip_grad_norm=1.0,
    )

    hist_exp = trainer_exp.fit(
        train_loader=train_loader,
        val_loader=val_loader,
        epochs=EPOCHS_LOSS_EXP,
        verbose=True,
    )

    final_val_mae = hist_exp['val_mae'][-1]
    final_val_rmse = hist_exp['val_rmse'][-1]
    final_val_r2 = hist_exp['val_r2'][-1]

    loss_experiment_results.append({
        'Loss Function': loss_name.upper(),
        'Final Train Loss': hist_exp['train_loss'][-1],
        'Final Val Loss': hist_exp['val_loss'][-1],
        'Val MAE': final_val_mae,
        'Val RMSE': final_val_rmse,
        'Val R2': final_val_r2,
    })

loss_df = pd.DataFrame(loss_experiment_results)
print('\n' + '=' * 75)
print('LOSS FUNCTION EXPERIMENT SUMMARY')
print('=' * 75)
print(loss_df.to_string(index=False))

# Section 5 — Optimizer Experiments

In [ ]:
optimizer_experiment_results = []
OPTIMIZER_CONFIGS = [
    ('Adam', 'adam', {'lr': 1e-3, 'weight_decay': 1e-4}),
    ('AdamW', 'adamw', {'lr': 1e-3, 'weight_decay': 1e-2}),
    ('SGD_Momentum', 'sgd', {'lr': 1e-2, 'momentum': 0.9, 'weight_decay': 1e-4, 'nesterov': True}),
]
EPOCHS_OPT_EXP = 2

for label, opt_name, opt_kwargs in OPTIMIZER_CONFIGS:
    print(f'\n--- Training MLP_Base with Optimizer: {label} ---')
    model_opt = build_model(mlp_base_config, device=DEVICE, seed=SEED)
    criterion_opt = get_loss_function('mse')
    optimizer_instance = get_optimizer(opt_name, model_opt, **opt_kwargs)

    trainer_opt = MLPTrainer(
        model=model_opt,
        optimizer=optimizer_instance,
        criterion=criterion_opt,
        device=DEVICE,
        clip_grad_norm=1.0,
    )

    hist_opt = trainer_opt.fit(
        train_loader=train_loader,
        val_loader=val_loader,
        epochs=EPOCHS_OPT_EXP,
        verbose=True,
    )

    optimizer_experiment_results.append({
        'Optimizer': label,
        'Initial LR': opt_kwargs['lr'],
        'Final Train Loss': hist_opt['train_loss'][-1],
        'Val Loss': hist_opt['val_loss'][-1],
        'Val MAE': hist_opt['val_mae'][-1],
        'Val RMSE': hist_opt['val_rmse'][-1],
        'Val R2': hist_opt['val_r2'][-1],
    })

opt_df = pd.DataFrame(optimizer_experiment_results)
print('\n' + '=' * 75)
print('OPTIMIZER EXPERIMENT SUMMARY')
print('=' * 75)
print(opt_df.to_string(index=False))

# Section 6 — Learning Rate Scheduling

In [ ]:
print('--- Evaluating ReduceLROnPlateau Scheduler ---')
model_plateau = build_model(mlp_base_config, device=DEVICE, seed=SEED)
opt_plateau = get_optimizer('adam', model_plateau, lr=1e-3, weight_decay=1e-4)
sched_plateau = get_scheduler('plateau', opt_plateau, mode='min', factor=0.5, patience=1, min_lr=1e-6)

trainer_plateau = MLPTrainer(
    model=model_plateau,
    optimizer=opt_plateau,
    criterion=get_loss_function('mse'),
    device=DEVICE,
)

hist_plateau = trainer_plateau.fit(
    train_loader=train_loader,
    val_loader=val_loader,
    epochs=3,
    scheduler=sched_plateau,
    verbose=True,
)

print(f'\nPlateau LR Progression: {[f"{lr:.2e}" for lr in hist_plateau["lr"]]}')

print('\n--- Evaluating CosineAnnealingLR Scheduler ---')
model_cosine = build_model(mlp_base_config, device=DEVICE, seed=SEED)
opt_cosine = get_optimizer('adam', model_cosine, lr=1e-3, weight_decay=1e-4)
sched_cosine = get_scheduler('cosine', opt_cosine, T_max=3, eta_min=1e-6)

trainer_cosine = MLPTrainer(
    model=model_cosine,
    optimizer=opt_cosine,
    criterion=get_loss_function('mse'),
    device=DEVICE,
)

hist_cosine = trainer_cosine.fit(
    train_loader=train_loader,
    val_loader=val_loader,
    epochs=3,
    scheduler=sched_cosine,
    verbose=True,
)

print(f'Cosine LR Progression : {[f"{lr:.2e}" for lr in hist_cosine["lr"]]}')

# Section 7 — Early Stopping & Checkpoint Restoration

In [ ]:
print('--- Demonstrating Early Stopping with Weight Restoration ---')
model_es = build_model(mlp_base_config, device=DEVICE, seed=SEED)
opt_es = get_optimizer('adam', model_es, lr=1e-3, weight_decay=1e-4)
crit_es = get_loss_function('mse')

early_stopping = EarlyStopping(patience=2, min_delta=0.01, mode='min', verbose=True)

trainer_es = MLPTrainer(
    model=model_es,
    optimizer=opt_es,
    criterion=crit_es,
    device=DEVICE,
)

hist_es = trainer_es.fit(
    train_loader=train_loader,
    val_loader=val_loader,
    epochs=4,
    early_stopping=early_stopping,
    verbose=True,
)

print(f'\nEarly Stopping Summary:')
print(f'  Best Epoch Recorded     : {early_stopping.best_epoch}')
print(f'  Best Validation Loss    : {early_stopping.best_score:.4f}')
print(f'  Early Stopped Triggered : {early_stopping.early_stop}')

val_after_restore = trainer_es.evaluate(val_loader)
print(f'  Validation Loss after Restoration: {val_after_restore["val_loss"]:.4f}')

# Section 8 — Training Monitoring & Convergence Curves

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

epochs_range = range(1, len(baseline_history['train_loss']) + 1)

axes[0, 0].plot(epochs_range, baseline_history['train_loss'], 'o-', label='Train Loss (MSE)', color='#1f77b4')
axes[0, 0].plot(epochs_range, baseline_history['val_loss'], 's--', label='Val Loss (MSE)', color='#ff7f0e')
axes[0, 0].set_title('Training & Validation Loss Convergence', fontsize=12, fontweight='bold')
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('Mean Squared Error')
axes[0, 0].legend()
axes[0, 0].grid(True, linestyle='--', alpha=0.6)

axes[0, 1].plot(epochs_range, baseline_history['train_mae'], 'o-', label='Train MAE', color='#2ca02c')
axes[0, 1].plot(epochs_range, baseline_history['val_mae'], 's--', label='Val MAE', color='#d62728')
axes[0, 1].set_title('Training vs Validation MAE', fontsize=12, fontweight='bold')
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('Mean Absolute Error (CCME points)')
axes[0, 1].legend()
axes[0, 1].grid(True, linestyle='--', alpha=0.6)

axes[1, 0].plot(epochs_range, baseline_history['lr'], 'd-', color='#9467bd', label='Learning Rate')
axes[1, 0].set_title('Learning Rate Trajectory', fontsize=12, fontweight='bold')
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].set_ylabel('Learning Rate')
axes[1, 0].set_yscale('log')
axes[1, 0].legend()
axes[1, 0].grid(True, linestyle='--', alpha=0.6)

axes[1, 1].plot(epochs_range, baseline_history['generalization_gap'], '^-', color='#8c564b', label='Generalization Gap (Val MAE - Train MAE)')
axes[1, 1].axhline(0, color='black', linestyle=':', alpha=0.7)
axes[1, 1].set_title('Generalization Gap Dynamics', fontsize=12, fontweight='bold')
axes[1, 1].set_xlabel('Epoch')
axes[1, 1].set_ylabel('Gap (CCME Index points)')
axes[1, 1].legend()
axes[1, 1].grid(True, linestyle='--', alpha=0.6)

plt.tight_layout()
figures_dir = PHASE2_DIR / 'reports' / 'figures'
figures_dir.mkdir(parents=True, exist_ok=True)
plt.savefig(figures_dir / '03_training_convergence_curves.png', dpi=300)
plt.close(fig)

print(f'Training curves figure saved → {figures_dir / "03_training_convergence_curves.png"}')

# Section 9 — Overfitting Analysis & Regularization Diagnostics

In [ ]:
overfit_diagnostics = analyze_overfitting(baseline_history)

print('=' * 75)
print('OVERFITTING & GENERALIZATION DIAGNOSTIC REPORT')
print('=' * 75)
for k, v in overfit_diagnostics.items():
    if isinstance(v, float):
        print(f'  {k:<30} : {v:.4f}')
    else:
        print(f'  {k:<30} : {v}')

print('\nFull Training Progress Table:')
print(format_history_table(baseline_history))

# Section 10 — Final Test Set Evaluation & Benchmark Comparison

In [ ]:
print('Evaluating final selected model on unseen test set...')
test_results = baseline_trainer.evaluate(test_loader)

test_mae = test_results['val_mae']
test_rmse = test_results['val_rmse']
test_r2 = test_results['val_r2']

print(f'\nFINAL TEST SET METRICS (MLP_Base):')
print(f'  Test MAE  : {test_mae:.4f}')
print(f'  Test RMSE : {test_rmse:.4f}')
print(f'  Test R2   : {test_r2:.4f}')

benchmark_data = [
    {
        'Phase': 'Phase 1',
        'Model Family': 'Mean Baseline',
        'Evaluation Split': 'Validation',
        'MAE': 16.7420,
        'RMSE': 20.3150,
        'R2': 0.0000,
        'Notes': 'Empirical mean floor benchmark'
    },
    {
        'Phase': 'Phase 1',
        'Model Family': 'Random Forest',
        'Evaluation Split': 'Validation',
        'MAE': 0.0521,
        'RMSE': 0.2081,
        'R2': 0.99986,
        'Notes': '30 trees, depth 14, min leaf 5'
    },
    {
        'Phase': 'Phase 1',
        'Model Family': 'XGBoost',
        'Evaluation Split': 'Validation',
        'MAE': 0.1611,
        'RMSE': 0.4372,
        'R2': 0.99940,
        'Notes': '150 trees, depth 6, lr 0.08 (unified QA)'
    },
    {
        'Phase': 'Phase 2',
        'Model Family': 'WaterQualityMLP (Base)',
        'Evaluation Split': 'Test (Unseen)',
        'MAE': test_mae,
        'RMSE': test_rmse,
        'R2': test_r2,
        'Notes': '4-layer funnel (512-256-128-64), BN, DO=0.3'
    },
]

benchmark_df = pd.DataFrame(benchmark_data)
print('\n' + '=' * 85)
print('CROSS-PHASE BENCHMARK COMPARISON TABLE')
print('=' * 85)
print(benchmark_df.to_string(index=False))

# Section 11 — Model Export & Resumable Checkpoint

In [ ]:
models_dir = PHASE2_DIR / 'models'
models_dir.mkdir(parents=True, exist_ok=True)
checkpoint_file = models_dir / 'mlp_best_checkpoint.pt'

save_training_checkpoint(
    path=checkpoint_file,
    model=baseline_model,
    optimizer=baseline_optimizer,
    epoch=EPOCHS_BASELINE,
    config=mlp_base_config,
    best_val_loss=min(baseline_history['val_loss']),
    history=baseline_history,
    extra_meta={
        'test_mae': test_mae,
        'test_rmse': test_rmse,
        'test_r2': test_r2,
        'feature_dimension': INPUT_DIM,
        'target': TARGET_VAR,
    }
)

reloaded_model, reloaded_meta = load_model_checkpoint(checkpoint_file, device=DEVICE)
print('\nTask 2 Checkpoint Loader Verification:')
print(f'  Model successfully reloaded  : {type(reloaded_model).__name__}')
print(f'  Reloaded Architecture Config : {reloaded_meta["architecture_config"]["name"]}')
print(f'  Reloaded Epoch                : {reloaded_meta["epoch"]}')
print(f'  Reloaded Test MAE             : {reloaded_meta["test_mae"]:.4f}')

history_json_path = PHASE2_DIR / 'configs' / 'training_history.json'
with open(history_json_path, 'w', encoding='utf-8') as f:
    json.dump({
        'model_name': 'MLP_Base',
        'history': baseline_history,
        'diagnostics': overfit_diagnostics,
        'test_metrics': {
            'mae': test_mae,
            'rmse': test_rmse,
            'r2': test_r2,
        }
    }, f, indent=2)

print(f'Training history JSON saved -> {history_json_path}')
print('\n* TASK 3: MLP TRAINING & OPTIMIZATION COMPLETED SUCCESSFULLY.')